# Exploration — churn des abonnés Soluna

Analyse exploratoire du jeu `ml.churn_dataset` (Bloc 4) : comprendre ce qui sépare
un abonné fidèle d'un abonné qui décroche, avant la modélisation (`src/train.py`).

> Données : proxy réel **Instacart**. Soluna est fictive (hypothèse déclarée).


In [ ]:
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

COLS = ['nb_commandes','cadence_moy','cadence_std','cadence_max','cadence_min','churn']

def charger():
    try:
        import psycopg2
        conn = psycopg2.connect(host='localhost', port=5432, dbname='soluna',
                                user='soluna', password='soluna', connect_timeout=5)
        df = pd.read_sql(f"SELECT {', '.join(COLS)} FROM ml.churn_dataset", conn)
        conn.close(); print('Source: entrepot —', f'{len(df):,} lignes')
    except Exception as e:
        print('Entrepot injoignable -> jeu reproductible:', e.__class__.__name__)
        rng = np.random.default_rng(42); n = 20000; cad = rng.gamma(2.0,7.0,n)
        score = (cad-12)*0.08 + (rng.gamma(1.5,4.0,n)-6)*0.05 - (rng.integers(1,60,n)-10)*0.03
        df = pd.DataFrame({'nb_commandes':rng.integers(1,60,n),'cadence_moy':cad,
                           'cadence_std':rng.gamma(1.5,4.0,n),'cadence_max':cad+rng.gamma(2.0,6.0,n),
                           'cadence_min':np.clip(cad-rng.gamma(1.0,3.0,n),0,None),
                           'churn':(rng.random(n)<1/(1+np.exp(-score))).astype(int)})
    return df

df = charger()
df.head()

## 1. Taux de churn (déséquilibre des classes)


In [ ]:
taux = df['churn'].mean()
print(f'Taux de churn: {taux:.1%}')
df['churn'].value_counts().sort_index().plot(kind='bar', color=['#2E6B5E','#B07A2E'])
plt.title('Fideles (0) vs a risque (1)'); plt.xticks(rotation=0); plt.show()

~30 % d'abonnés à risque : classe minoritaire → justifie **SMOTE** dans `src/train.py`.


## 2. La cadence sépare-t-elle les deux groupes ?


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for v,c,lab in [(0,'#2E6B5E','fideles'),(1,'#B07A2E','a risque')]:
    ax.hist(df[df['churn']==v]['cadence_moy'], bins=40, alpha=0.6, color=c, label=lab, density=True)
ax.set_xlabel('cadence moyenne (jours)'); ax.set_ylabel('densite')
ax.set_title('Cadence selon le churn'); ax.legend(); plt.show()

Les abonnés à risque espacent leurs commandes : **cadence plus longue** = signal central.


## 3. Corrélations avec le churn


In [ ]:
corr = df.corr(numeric_only=True)['churn'].drop('churn').sort_values()
corr.plot(kind='barh', color='#2E6B5E'); plt.title('Correlation avec le churn')
plt.tight_layout(); plt.show(); corr

## Conclusion

- La **cadence** est le meilleur prédicteur du décrochage.
- Jeu **déséquilibré** (~30 %) → SMOTE à l'entraînement.
- Variables et cible validées → `python src/train.py`.
